
# Lab #7: Keras MLP for Multiclass Classification
## Mental Health / Student Stress Dataset

### Objective
Implement a Multi-Layer Perceptron (MLP) using Keras/TensorFlow for a real-world multiclass classification problem and compare different activation functions, optimizers, and model configurations.

### Selected problem
**Multiclass classification of student stress severity** into:

- Low
- Medium
- High

The selected dataset is the **Student Stress Factors Dataset**, containing student-related psychological, physiological, environmental, academic, and social factors.

> **Important:** This model is intended for educational/research classification of stress severity and is **not a clinical diagnostic system**.



## 1. Why this dataset?

This dataset is suitable for the practical because:

- It is related to student mental-health/stress factors.
- The target is naturally multiclass.
- It contains three stress categories: **Low, Medium, High**.
- The classes are approximately balanced.
- It contains a mixture of psychological, physiological, environmental, academic, and social factors.
- It allows meaningful interpretation of misclassification.
- It is suitable for comparing ReLU, Sigmoid, and Tanh.
- It is suitable for comparing SGD, Adam, and RMSprop.

### Source
Student Stress Factors Dataset:
https://www.kaggle.com/datasets/samyakb/student-stress-factors

A 2026 peer-reviewed study describing the dataset reports approximately **1,100 student records and 20 features**, with stress severity represented by three classes.

**Use the downloaded CSV from the dataset source in the upload cell below.**



## 2. Regression vs Classification

| Aspect | Regression | Classification |
|---|---|---|
| Output | Continuous value | Class/category |
| Example | Predict SGPA | Predict stress level |
| Target | 7.8 | High |
| Typical output | Linear | Sigmoid/Softmax |
| Common losses | MSE, MAE | Cross-entropy |
| Metrics | MAE, RMSE, R² | Accuracy, Precision, Recall, F1 |

**Regression answers:** “How much?”

**Classification answers:** “Which class?”

This experiment is a **multiclass classification** problem because the target contains three categories.



## 3. MLP architecture

An MLP consists of:

1. Input layer
2. At least two hidden layers
3. Output layer

General structure:

**Input features → Dense/ReLU → Dense/ReLU → Dense/ReLU → Softmax output**

For three classes, the output layer contains **three neurons**.

The Softmax output represents the probability of each class.



## 4. Activation functions

### ReLU
`f(x) = max(0, x)`

Usually effective in hidden layers because it introduces non-linearity and is computationally simple.

### Sigmoid
Outputs values between 0 and 1.

Useful in binary-output settings, but included here as a required hidden-layer comparison.

### Tanh
Outputs values between -1 and +1.

Can provide nonlinear transformations but may behave differently from ReLU because of saturation.

### Softmax
Used in the final multiclass output layer to convert logits into class probabilities whose sum is 1.



## 5. Loss function

For integer-encoded multiclass labels, use:

`Sparse Categorical Cross-Entropy`

It penalizes predictions according to the probability assigned to the correct class.

Important distinction:

- **Accuracy:** proportion of correctly classified observations.
- **Loss:** degree of disagreement between predicted probabilities and actual labels.

A model can have similar accuracy but different loss because loss also considers prediction confidence.



## 6. Experimental design

### Experiment A — Hidden activation comparison

Keep the dataset split, architecture, optimizer, epochs, batch size, and other major hyperparameters consistent.

| Model | Hidden activation | Optimizer |
|---|---|---|
| Model 1 | ReLU | Adam |
| Model 2 | Sigmoid | Adam |
| Model 3 | Tanh | Adam |

### Experiment B — Optimizer comparison

Keep ReLU as the hidden activation.

| Model | Hidden activation | Optimizer |
|---|---|---|
| Model 4 | ReLU | SGD |
| Model 5 | ReLU | RMSprop |

Compare:

- Training accuracy
- Validation accuracy
- Training loss
- Validation loss
- Test accuracy
- Precision
- Recall
- F1-score
- Convergence behavior
- Generalization
- Confusion matrix



## 7. Real-world relevance

Possible applications of a similar classification approach include:

- Student support and early-warning systems
- Academic counselling prioritization
- Wellness program targeting
- Student engagement analysis
- Population-level stress monitoring
- Educational resource allocation

The predictions should be treated as **screening/research signals rather than medical diagnoses**.


In [ ]:

# Install/import required libraries

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)



## 8. Load the dataset

Upload the CSV downloaded from Kaggle/your approved source into Colab.

The following cell automatically looks for CSV files. If multiple files are present, inspect the filenames and select the correct one.


In [ ]:

from google.colab import files

uploaded = files.upload()

csv_files = [f for f in uploaded.keys() if f.lower().endswith(".csv")]
print("CSV files found:", csv_files)

if not csv_files:
    raise FileNotFoundError("Please upload the Student Stress Factors CSV file.")

DATA_PATH = csv_files[0]
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:

# Basic exploration

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())



## 9. Identify the target column

Because dataset versions can use slightly different column names, the next cell searches for common stress-target names.

If the automatically detected column is incorrect, manually set `TARGET_COLUMN`.


In [ ]:

# Detect likely target column

target_candidates = [
    c for c in df.columns
    if any(term in c.lower() for term in [
        "stress level", "stress_level", "stresslevel", "stress"
    ])
]

print("Possible target columns:", target_candidates)

# Change this manually if necessary.
TARGET_COLUMN = target_candidates[-1] if target_candidates else None

print("Selected target:", TARGET_COLUMN)

if TARGET_COLUMN is None:
    raise ValueError(
        "Target column was not automatically detected. "
        "Set TARGET_COLUMN to the actual stress-level column name."
    )


In [ ]:

# Inspect target distribution

print("Target values:")
display(df[TARGET_COLUMN].value_counts(dropna=False))

print("\nTarget proportions:")
display(df[TARGET_COLUMN].value_counts(normalize=True, dropna=False).mul(100).round(2))

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x=TARGET_COLUMN)
plt.title("Distribution of Student Stress Classes")
plt.xlabel("Stress Level")
plt.ylabel("Number of Students")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()



### Observation / interpretation template

**Observation:** The class-distribution graph shows the number of observations belonging to each stress category.

**Interpretation:** A relatively balanced distribution reduces the risk that the MLP simply favors a dominant class.

**Justification:** Balanced classes make accuracy more informative, although precision, recall, F1-score, and the confusion matrix should still be examined.


In [ ]:

# Remove rows with missing target
df = df.dropna(subset=[TARGET_COLUMN]).copy()

X = df.drop(columns=[TARGET_COLUMN])
y_raw = df[TARGET_COLUMN].astype(str).str.strip()

# Encode target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print("Classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

if len(label_encoder.classes_) < 3:
    raise ValueError(
        "The selected target must contain at least 3 classes for this lab."
    )



## 10. Feature preprocessing

The preprocessing pipeline:

1. Separate numerical and categorical variables.
2. Impute missing values.
3. One-hot encode categorical variables.
4. Standardize numerical features.
5. Split the data into training and testing sets.

Scaling is important for MLPs because features measured on very different numerical scales can affect optimization and convergence.


In [ ]:

from sklearn.preprocessing import OneHotEncoder

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)



## 11. MLP model definition

The architecture below uses at least two hidden layers as required.

- Input layer: number of processed features
- Hidden layer 1: 64 neurons
- Hidden layer 2: 32 neurons
- Hidden layer 3: 16 neurons
- Output layer: number of classes
- Output activation: Softmax


In [ ]:

N_FEATURES = X_train.shape[1]
N_CLASSES = len(label_encoder.classes_)

def build_mlp(hidden_activation="relu"):
    model = Sequential([
        Input(shape=(N_FEATURES,)),
        Dense(64, activation=hidden_activation),
        Dense(32, activation=hidden_activation),
        Dense(16, activation=hidden_activation),
        Dense(N_CLASSES, activation="softmax")
    ])
    return model

def compile_model(model, optimizer_name="adam", learning_rate=None):
    if optimizer_name.lower() == "adam":
        optimizer = Adam(learning_rate=learning_rate) if learning_rate else Adam()
    elif optimizer_name.lower() == "sgd":
        optimizer = SGD(learning_rate=learning_rate) if learning_rate else SGD()
    elif optimizer_name.lower() == "rmsprop":
        optimizer = RMSprop(learning_rate=learning_rate) if learning_rate else RMSprop()
    else:
        raise ValueError("Unsupported optimizer.")

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

print("Input features:", N_FEATURES)
print("Number of classes:", N_CLASSES)



## 12. Train activation-function experiments

For a fair comparison, use the same train/test split and similar training settings for every model.


In [ ]:

EPOCHS = 60
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.20

activation_results = {}
activation_histories = {}

for activation in ["relu", "sigmoid", "tanh"]:
    print(f"\nTraining activation: {activation}")

    model = build_mlp(hidden_activation=activation)
    compile_model(model, optimizer_name="adam")

    history = model.fit(
        X_train, y_train,
        validation_split=VALIDATION_SPLIT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    activation_results[activation] = {
        "model": model,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "y_pred": y_pred,
        "y_prob": y_prob
    }
    activation_histories[activation] = history

activation_summary = pd.DataFrame({
    activation: {
        "Test Accuracy": result["test_accuracy"],
        "Macro Precision": result["precision_macro"],
        "Macro Recall": result["recall_macro"],
        "Macro F1": result["f1_macro"]
    }
    for activation, result in activation_results.items()
}).T

display(activation_summary.sort_values("Macro F1", ascending=False))


In [ ]:

# Plot validation accuracy for activation functions

plt.figure(figsize=(10, 6))

for activation, history in activation_histories.items():
    plt.plot(history.history["val_accuracy"], label=f"{activation} validation")

plt.title("Validation Accuracy: Activation Function Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Plot validation loss for activation functions

plt.figure(figsize=(10, 6))

for activation, history in activation_histories.items():
    plt.plot(history.history["val_loss"], label=f"{activation} validation")

plt.title("Validation Loss: Activation Function Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



### Activation-function analysis

Use the actual results to answer:

1. Which activation function achieved the highest validation performance?
2. Which activation function achieved the highest test F1-score?
3. Which activation converged fastest?
4. Did any activation show unstable validation loss?
5. Did the training/validation gap indicate overfitting?

**Do not assume ReLU is best before examining the experimental evidence.**



## 13. Optimizer comparison

Now compare:

- Adam
- SGD
- RMSprop

The hidden activation is fixed at **ReLU** so that the main experimental variable is the optimizer.


In [ ]:

optimizer_results = {}
optimizer_histories = {}

for optimizer_name in ["adam", "sgd", "rmsprop"]:
    print(f"\nTraining optimizer: {optimizer_name}")

    model = build_mlp(hidden_activation="relu")
    compile_model(model, optimizer_name=optimizer_name)

    history = model.fit(
        X_train, y_train,
        validation_split=VALIDATION_SPLIT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    optimizer_results[optimizer_name] = {
        "model": model,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "y_pred": y_pred,
        "y_prob": y_prob
    }
    optimizer_histories[optimizer_name] = history

optimizer_summary = pd.DataFrame({
    optimizer: {
        "Test Accuracy": result["test_accuracy"],
        "Macro Precision": result["precision_macro"],
        "Macro Recall": result["recall_macro"],
        "Macro F1": result["f1_macro"]
    }
    for optimizer, result in optimizer_results.items()
}).T

display(optimizer_summary.sort_values("Macro F1", ascending=False))


In [ ]:

# Optimizer validation accuracy

plt.figure(figsize=(10, 6))

for optimizer_name, history in optimizer_histories.items():
    plt.plot(history.history["val_accuracy"], label=f"{optimizer_name} validation")

plt.title("Validation Accuracy: Optimizer Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Optimizer validation loss

plt.figure(figsize=(10, 6))

for optimizer_name, history in optimizer_histories.items():
    plt.plot(history.history["val_loss"], label=f"{optimizer_name} validation")

plt.title("Validation Loss: Optimizer Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



### Optimizer analysis

Discuss:

- Which optimizer converged fastest?
- Which produced the lowest validation loss?
- Which produced the highest test accuracy?
- Which produced the best macro F1-score?
- Which optimizer showed the most stable learning curve?
- Did the optimizer materially affect generalization?

Again, base the final conclusion on the measured results.


In [ ]:

# Combined comparison table required by the lab

comparison_rows = []

for activation, result in activation_results.items():
    comparison_rows.append({
        "Experiment": f"Activation - {activation}",
        "Activation": activation,
        "Optimizer": "Adam",
        "Test Accuracy": result["test_accuracy"],
        "Precision (Macro)": result["precision_macro"],
        "Recall (Macro)": result["recall_macro"],
        "F1-Score (Macro)": result["f1_macro"]
    })

for optimizer_name, result in optimizer_results.items():
    comparison_rows.append({
        "Experiment": f"Optimizer - {optimizer_name}",
        "Activation": "ReLU",
        "Optimizer": optimizer_name,
        "Test Accuracy": result["test_accuracy"],
        "Precision (Macro)": result["precision_macro"],
        "Recall (Macro)": result["recall_macro"],
        "F1-Score (Macro)": result["f1_macro"]
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.sort_values("F1-Score (Macro)", ascending=False).reset_index(drop=True))



## 14. Select the final model

The final model should not be selected using accuracy alone.

Consider:

- Test accuracy
- Macro precision
- Macro recall
- Macro F1-score
- Confusion matrix
- Training/validation curves
- Convergence
- Generalization
- Model complexity

The lab specifically requires a justified final-model selection based on experimental evidence.


In [ ]:

# Automatically select the best model using macro F1 as the primary criterion.
# Accuracy is used as a secondary criterion.

all_models = []

for activation, result in activation_results.items():
    all_models.append({
        "name": f"{activation} + Adam",
        "model": result["model"],
        "y_pred": result["y_pred"],
        "f1": result["f1_macro"],
        "accuracy": result["test_accuracy"],
        "precision": result["precision_macro"],
        "recall": result["recall_macro"]
    })

for optimizer_name, result in optimizer_results.items():
    all_models.append({
        "name": f"ReLU + {optimizer_name}",
        "model": result["model"],
        "y_pred": result["y_pred"],
        "f1": result["f1_macro"],
        "accuracy": result["test_accuracy"],
        "precision": result["precision_macro"],
        "recall": result["recall_macro"]
    })

best_model_info = sorted(
    all_models,
    key=lambda x: (x["f1"], x["accuracy"]),
    reverse=True
)[0]

print("Selected final model:", best_model_info["name"])
print("Test accuracy:", round(best_model_info["accuracy"], 4))
print("Macro precision:", round(best_model_info["precision"], 4))
print("Macro recall:", round(best_model_info["recall"], 4))
print("Macro F1:", round(best_model_info["f1"], 4))


In [ ]:

# Detailed classification report

final_y_pred = best_model_info["y_pred"]

print(classification_report(
    y_test,
    final_y_pred,
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0
))


In [ ]:

# Confusion matrix

cm = confusion_matrix(y_test, final_y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title(f"Confusion Matrix — {best_model_info['name']}")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()



## 15. Confusion-matrix interpretation

### Observation
The diagonal cells represent correctly classified students. Off-diagonal cells represent misclassifications.

### Interpretation
The class with the largest diagonal value has the largest number of correct predictions in absolute terms.

### Difficult class pair
Inspect the largest off-diagonal value to identify which two stress categories are most frequently confused.

### Possible reasons for misclassification

- Overlap between stress categories
- Similar psychological/academic profiles
- Self-reported measurement noise
- Correlated features
- Limited sample size
- Features that do not fully capture individual differences
- Borderline cases between adjacent stress levels

Do not claim a specific psychological cause unless supported by the dataset or literature.


In [ ]:

# Training/validation curves for the selected model
# Find its corresponding history

selected_name = best_model_info["name"]

if selected_name.endswith("+ Adam"):
    selected_activation = selected_name.split(" + ")[0]
    selected_history = activation_histories[selected_activation]
else:
    selected_optimizer = selected_name.split(" + ")[1]
    selected_history = optimizer_histories[selected_optimizer]

fig = plt.figure(figsize=(10, 6))
plt.plot(selected_history.history["accuracy"], label="Training accuracy")
plt.plot(selected_history.history["val_accuracy"], label="Validation accuracy")
plt.title(f"Accuracy Curves — {selected_name}")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(10, 6))
plt.plot(selected_history.history["loss"], label="Training loss")
plt.plot(selected_history.history["val_loss"], label="Validation loss")
plt.title(f"Loss Curves — {selected_name}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 16. Observation, Interpretation and Justification

Use this structure in the final lab record.

### Observation
State exactly what the graph/table shows.

Example:
> Training accuracy increased steadily, while validation accuracy increased initially and then stabilized.

### Interpretation
Explain what the observation means.

Example:
> This indicates that the model learned useful patterns from the training data and eventually reached a relatively stable validation performance.

### Justification
Explain why the interpretation is reasonable.

Example:
> The conclusion is supported by the simultaneous reduction in training loss and stabilization of validation loss without a large persistent train-validation gap.

### Result
Report the actual measured values:

- Test accuracy
- Macro precision
- Macro recall
- Macro F1
- Important confusion-matrix findings



## 17. Key viva questions

### Q1. Why is this classification rather than regression?
Because the target is a discrete stress category: Low, Medium, or High.

### Q2. Why is Softmax used in the output layer?
It converts output scores into probabilities across mutually exclusive classes.

### Q3. Why use sparse categorical cross-entropy?
The target labels are integer encoded rather than one-hot encoded.

### Q4. Why scale numerical features?
MLPs are trained through gradient-based optimization, and feature-scale differences can adversely affect optimization.

### Q5. Why compare ReLU, Sigmoid and Tanh?
They introduce non-linearity differently and can produce different gradient behavior and convergence characteristics.

### Q6. Why compare Adam, SGD and RMSprop?
Optimizers use different parameter-update strategies and can differ in convergence speed, stability, and generalization.

### Q7. What indicates overfitting?
Training performance continues improving while validation performance deteriorates or validation loss increases.

### Q8. What does the confusion matrix show?
It shows actual versus predicted class counts and identifies class-specific errors.

### Q9. Why use macro-average metrics?
Macro averaging gives each class equal importance, which is useful when we want to assess performance across all stress categories rather than allowing a larger class to dominate the metric.

### Q10. Why should accuracy not be the only selection criterion?
A model can have high accuracy while performing poorly for one class. Precision, recall, F1-score, confusion matrix, convergence, and generalization provide a broader assessment.



## 18. Final conclusion template

> The Keras MLP was implemented for three-class student stress classification using psychological, physiological, environmental, academic, and social features. Multiple hidden-layer activation functions and optimizers were experimentally compared using the same dataset split and evaluation procedure. Performance was assessed using training/validation accuracy and loss, test accuracy, macro precision, macro recall, macro F1-score, and the confusion matrix. The final model was selected based on overall classification performance, convergence behavior, and generalization rather than accuracy alone. The experimental results demonstrate how activation functions and optimizers can influence MLP convergence and classification performance.



## References

1. Student Stress Factors Dataset, Kaggle:
   https://www.kaggle.com/datasets/samyakb/student-stress-factors

2. Frontiers in Computer Science (2026), study describing the Student Stress Factors dataset and its three-class stress severity formulation:
   https://www.frontiersin.org/journals/computer-science/articles/10.3389/fcomp.2026.1886274/full

3. Keras documentation:
   https://keras.io/

4. TensorFlow documentation:
   https://www.tensorflow.org/

### Lab requirements incorporated

This notebook covers the requirements in the provided Lab #7 handout: dataset description/source, multiclass target, preprocessing, MLP architecture with at least two hidden layers, activation-function comparison, optimizer comparison, training/validation plots, multiclass metrics, confusion matrix, comparison table, interpretation, justification, and final model selection.
